# P12: answer five questions about coco_pharma's sales tables, with and without the context pack

The first measurement of `docs/context-as-product-requirements.md` P12, as proposed in
`docs/context-pack-first-instance-coco-pharma.md` section 5. Write-up: `docs/experiments/p12-results.md`.

**What this does.** Builds the pack with `trellis_analytics.routines.context_pack` (it reads Egeria only, as RE's daemon
identity, read-only), then asks RE's configured local model the same five fixed questions twice:
**without** the pack (the question and the subject name only) and **with** it (the pack's manifest and documents
in front of the question). A third condition, **with + rules**, adds one paragraph telling the model how to read the
pack's state words; it is reported separately so the effect of the pack and the effect of the instruction are not
mixed. Each answer is scored by a rubric written below, with no model in the scoring.

**Read-only.** Egeria reads go through the MetadataExpert client's read methods; the registry is read with a read-only
Postgres session (SELECT only) for the *names* of what RE holds and has not published (never their values).
The coco_pharma source database is not queried at all: the pack comes from what Egeria and RE hold.

In [1]:
subject_guid = "0ff72014-8936-4bea-b3d1-6a4493b99b30"     # the coco_pharma database element in Egeria (post-wipe)
subject_slug = "localhost_docker_coco_pharma"            # RE's slug for it (read-only, for the names RE holds)
subject_name = "coco_pharma"
consumer_kind = "data scientist"
schemas = ["us_sales", "target_sales"]                   # the sales schemas; the other catalogued schemas are counted as left out
library_src = ""                                         # a path to prepend to sys.path (a worktree's trellis-analytics/src), or empty
out_dir = "."                                            # where p12-pack.json and p12-outputs.json are written
model = ""                                               # empty: RE's configured Ollama model
temperature = 0.0                                        # fixed for repeatability (RE's chat default is 0.1)
seed = 7

In [2]:
# Parameters
library_src = "/Users/dwolfson/localGit/egeria-v6/wt-p12/packages/trellis-analytics/src"
out_dir = "/Users/dwolfson/localGit/egeria-v6/wt-p12/docs/experiments"


In [3]:
import sys, json, inspect, datetime
if library_src:
    sys.path.insert(0, library_src)
import trellis_analytics, trellis_analytics.answer_scoring as scoring
from trellis_analytics.readers.egeria import EgeriaPackReader
from trellis_analytics.routines.context_pack import context_pack, ReHeld
print("trellis_analytics from", trellis_analytics.__file__)

trellis_analytics from /Users/dwolfson/localGit/egeria-v6/wt-p12/packages/trellis-analytics/src/trellis_analytics/__init__.py


## 1. What RE holds and has not published (names only)
The pack never reads RE. This cell reads the registry *read-only* and passes the routine only the **names** of facts RE holds, so the routine can say "not published" for them and quote no value.

In [4]:
import psycopg2
from resource_explorer.config import RegistryConfig
conn = psycopg2.connect(RegistryConfig().database_url); conn.set_session(readonly=True, autocommit=True)
cur = conn.cursor()
ph = ",".join(["%s"] * len(schemas))
cur.execute(f"select distinct schema_name, table_name from resource_explorer.database_tables "
            f"where database_slug=%s and schema_name in ({ph}) and row_count is not null", (subject_slug, *schemas))
row_count_tables = {f"{s}.{t}" for s, t in cur.fetchall()}
cur.execute(f"select distinct schema_name, table_name from resource_explorer.database_column_profiles "
            f"where database_slug=%s and schema_name in ({ph})", (subject_slug, *schemas))
profile_tables = {f"{s}.{t}" for s, t in cur.fetchall()}
# schemas RE knows that Egeria's record may not hold, with the newest scope choice as the reason (read below against Egeria's list)
cur.execute("select distinct schema_name from resource_explorer.database_schemas where database_slug=%s", (subject_slug,))
re_schemas = sorted(r[0] for r in cur.fetchall())
cur.execute("select distinct on (schema_name) schema_name, choice, source from resource_explorer.catalogue_scope_events "
            "where database_slug=%s and node_kind='schema' order by schema_name, id desc", (subject_slug,))
choices = {n: (c, s) for n, c, s in cur.fetchall()}
conn.close()
print("RE holds a row count for:", sorted(row_count_tables)); print("RE holds column profiles for:", sorted(profile_tables))
print(len(re_schemas), "schemas known to RE")

RE holds a row count for: ['target_sales.consolidated_forecast', 'us_sales.us_sales_forecast']
RE holds column profiles for: ['target_sales.consolidated_forecast', 'us_sales.us_sales_forecast']
30 schemas known to RE


## 2. Build the pack from Egeria

In [5]:
from resource_explorer.egeria_clients import egeria_client, Daemon, DaemonReason
from pyegeria.omvs.metadata_expert import MetadataExpert
ident = Daemon(DaemonReason.BOOTSTRAP)
reader = EgeriaPackReader(egeria_client(ident, purpose="read-only P12 context pack").of(MetadataExpert), ident.user_id)

def why(name):
    c = choices.get(name)
    if c is None: return "undecided in the scope"
    return {"catalogue": "chosen for the catalogue, not in Egeria's record",
            "leave_out": "left out of the scope" + (" by a person" if c[1] == "person" else f" ({c[1]})")}.get(c[0], f"scope choice {c[0]}")

# first build with no schema reasons, to learn which schemas Egeria's record holds; then name the rest
probe = context_pack(reader, subject_guid, consumer_kind, ident.user_id, schemas=schemas)
outside = {n for l in probe.manifest.left_out if l.what.startswith("schemas outside") for n in l.names}
in_egeria = outside | set(schemas)      # every schema Egeria's record holds: the pack's, plus the ones scope left out
held = ReHeld(fact_kinds={"row_count": row_count_tables, "column_profile": profile_tables},
              schemas_not_in_egeria={n: why(n) for n in re_schemas if n not in in_egeria})
pack = context_pack(reader, subject_guid, consumer_kind, ident.user_id, schemas=schemas, re_held=held)
print(pack.envelope.result_state.words, pack.envelope.result_state_reason)
print(pack.manifest.text)

measured 
Pack: coco_pharma · data scientist · v1 · as of 2026-10-14T00:50:42Z · built for resourceexplorernpa
Included: 1 databases · 2 schemas · 2 tables · 21 columns · 1 owners · 2 survey reports · 10 survey annotations used · 3 documents
Left out and why:
  2 schemas outside this pack's scope · left out · Egeria records them; this pack was asked for other schemas
  1 schemas not in Egeria: left out of the scope (no_writes) · not published · RE knows these schemas; Egeria's record does not hold them (left out of the scope (no_writes))
  25 schemas not in Egeria: undecided in the scope · not published · RE knows these schemas; Egeria's record does not hold them (undecided in the scope)
  2 row counts · not published · measured by RE, not published to Egeria; the pack quotes none
  2 column profiles · not published · measured by RE, not published to Egeria
  13 column data types · not established · the cataloguer records no data type; only columns an Egeria survey profiled have one
  

In [6]:
pack_dict = pack.to_dict()
json.dump(pack_dict, open(f"{out_dir}/p12-pack.json", "w"), indent=1)
for t in pack.tabular:
    print(f"{t.schema_name}.{t.table}: columns {t.columns.value} ({t.columns.words}) | rows: {t.rows.words} | size {t.size_bytes.value} ({t.size_bytes.words}) | term: {t.term.words} | owner {t.owner.value} | licence: {t.licence.words}")
print(pack.notes)

target_sales.consolidated_forecast: columns 10 (recorded) | rows: not published | size 0 (measured) | term: nothing found | owner Jules | licence: not established
us_sales.us_sales_forecast: columns 11 (recorded) | rows: not published | size 147456 (measured) | term: nothing found | owner Jules | licence: not established
["schema us_sales was not in the database's relationship list; found by name search of its qualified name"]


## 3. The five fixed questions

Fixed before the run; they are about the sales tables Egeria's cataloguer holds (`us_sales.us_sales_forecast`, `target_sales.consolidated_forecast`).
`expects_state` lists the pack's own phrases, any of which earns the state-word axis; an empty list means the pack has no gap there, so that axis is not scored.
`key_facts` are read from the record (not from the source database) and report whether the answer was also right; they are not an axis.

In [7]:
Q = scoring.Question
questions = [
  Q("Q1", "Which tables in the coco_pharma database hold sales data, and which schemas are they in?",
    key_facts=[["us_sales_forecast"], ["consolidated_forecast"], ["us_sales"], ["target_sales"]]),
  Q("Q2", "How many rows are in the us_sales_forecast table?",
    expects_state=["not published", "not established"], key_facts=[["not published", "not established"]]),
  Q("Q3", "What columns does the us_sales_forecast table have, and what is the data type of its forecastamount column?",
    key_facts=[["forecastamount"], ["opportunityid"], ["closedate"], ["numeric"]]),
  Q("Q4", "Who owns the us_sales_forecast table, and which business glossary term is it assigned to?",
    expects_state=["nothing found", "not published", "not established"], key_facts=[["jules"], ["nothing found"]]),
  Q("Q5", "How large is the consolidated_forecast table in the target_sales schema, and can I rely on that figure? Is there a licence or lineage recorded for it?",
    expects_state=["not established"], key_facts=[["0 bytes", "size in bytes: 0", "zero"], ["not established"], ["disagree", "do not agree", "questionable", "does not match", "doesn't match"]]),
]
for q in questions: print(q.id, q.text)

Q1 Which tables in the coco_pharma database hold sales data, and which schemas are they in?
Q2 How many rows are in the us_sales_forecast table?
Q3 What columns does the us_sales_forecast table have, and what is the data type of its forecastamount column?
Q4 Who owns the us_sales_forecast table, and which business glossary term is it assigned to?
Q5 How large is the consolidated_forecast table in the target_sales schema, and can I rely on that figure? Is there a licence or lineage recorded for it?


## 4. The rubric
The scorer is `trellis_analytics.answer_scoring`; its source is printed so the rubric sits beside the scores.

In [8]:
print(inspect.getsource(scoring))

"""The P12 rubric: score one answer, automatically and transparently.

Used by docs/experiments/p12-context-pack-coco-pharma.ipynb, which prints this file so the rubric a score came from
is on the page. Three axes, each 0 or 1, none judged by a model:

1. cites_record: the answer contains at least one element GUID that is in the pack (a full GUID, compared
   case-insensitively). A table name alone does not count: the question already names the tables.
2. state_words: where the pack carries a gap for the thing asked (`Question.expects_state`), the answer contains
   one of the pack's own state phrases ("not published", "not established", "nothing found", "not applicable").
   `None` when the pack has no gap there, so the axis is not scored. `admits_unknown` is reported beside it, lenient
   wording ("cannot", "don't have"...), to show a hedge that used other words.
3. no_fabrication: the answer contains no number the pack does not contain (GUIDs and timestamps removed from
   both befo

## 5. Ask RE's configured local model

In [9]:
import ollama
from resource_explorer.config import get_config
cfg = get_config().llm.ollama
llm_model = model or cfg.model
client = ollama.Client(host=cfg.base_url)
def ask(system, user):
    r = client.chat(model=llm_model, messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
                    options={"temperature": temperature, "num_ctx": cfg.num_ctx, "seed": seed})
    return r["message"]["content"]

pack_text = pack.manifest.text + "\n\n" + "\n\n".join(d.text for d in pack.documents)
pack_guids = scoring.guids_in(pack_text)          # every GUID the pack text shows: elements, columns, annotations
pack_names = {f"{subject_name}.{t.schema_name}.{t.table}" for t in pack.tabular}
BASE = "You answer questions from a data scientist about a database. Be concise."
RULES = ("Use only the context pack. Cite the element GUID of the record each fact comes from. Where the pack says a fact is "
         "'not published', 'not established' or 'nothing found', say exactly that; do not estimate or guess a value.")
def prompts(q, cond):
    ask_q = f"Database: {subject_name}\nQuestion: {q.text}"
    if cond == "without":      return BASE, ask_q
    ctx = f"Context pack (what the metadata catalogue holds):\n\n{pack_text}\n\n"
    if cond == "with":         return BASE + " Use the context pack.", ctx + ask_q
    return BASE + " " + RULES, ctx + ask_q
print("model:", llm_model, "at", cfg.base_url, "| pack chars:", len(pack_text))

model: llama3.1:8b at http://localhost:11434 | pack chars: 11191


In [10]:
results = []
for q in questions:
    for cond in ("without", "with", "with_rules"):
        sys_p, user_p = prompts(q, cond)
        ans = ask(sys_p, user_p)
        sc = scoring.score_answer(ans, q, pack_text, pack_guids, pack_names)
        results.append({"question": q.id, "condition": cond, "prompt_chars": len(sys_p) + len(user_p), "answer": ans, "score": sc})
        print(f"{q.id} {cond:10} cites={sc['cites_record']} state={sc['state_words']} nofab={sc['no_fabrication']} "
              f"names={sc['names_record']} total={sc['total']}/{sc['out_of']} facts={sc['key_facts_hit']} fab={sc['fabricated']}")

Q1 without    cites=0 state=None nofab=1 names=False total=1/2 facts=0/4 fab={'numbers': [], 'guids': []}


Q1 with       cites=0 state=None nofab=1 names=True total=1/2 facts=4/4 fab={'numbers': [], 'guids': []}


Q1 with_rules cites=0 state=None nofab=1 names=False total=1/2 facts=4/4 fab={'numbers': [], 'guids': []}


Q2 without    cites=0 state=0 nofab=0 names=False total=0/3 facts=0/1 fab={'numbers': ['1200'], 'guids': []}


Q2 with       cites=0 state=1 nofab=1 names=False total=2/3 facts=1/1 fab={'numbers': [], 'guids': []}


...

Q4 with       cites=0 state=0 nofab=1 names=False total=1/3 facts=1/2 fab={'numbers': [], 'guids': []}


Q4 with_rules cites=1 state=1 nofab=1 names=False total=3/3 facts=2/2 fab={'numbers': [], 'guids': []}


Q5 without    cites=0 state=0 nofab=0 names=True total=0/3 facts=0/3 fab={'numbers': ['123456'], 'guids': []}


Q5 with       cites=0 state=0 nofab=1 names=False total=1/3 facts=2/3 fab={'numbers': [], 'guids': []}


Q5 with_rules cites=1 state=1 nofab=1 names=False total=3/3 facts=3/3 fab={'numbers': [], 'guids': []}


In [11]:
summary = {}
for cond in ("without", "with", "with_rules"):
    rs = [r["score"] for r in results if r["condition"] == cond]
    st = [r["state_words"] for r in rs if r["state_words"] is not None]
    summary[cond] = {"cites_record": f"{sum(r['cites_record'] for r in rs)}/{len(rs)}",
                     "state_words": f"{sum(st)}/{len(st)}",
                     "no_fabrication": f"{sum(r['no_fabrication'] for r in rs)}/{len(rs)}",
                     "total": f"{sum(r['total'] for r in rs)}/{sum(r['out_of'] for r in rs)}",
                     "answers_with_fabricated_numbers": sum(1 for r in rs if r["fabricated"]["numbers"]),
                     "names_record (info)": f"{sum(r['names_record'] for r in rs)}/{len(rs)}"}
out = {"meta": {"run_at": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "model": llm_model, "ollama": cfg.base_url,
                "temperature": temperature, "seed": seed, "num_ctx": cfg.num_ctx, "identity": ident.user_id,
                "pack_as_of": pack.as_of, "pack_envelope": pack.envelope.result_state.value, "subject_guid": subject_guid, "schemas": schemas,
                "egeria_reads": "read-only (MetadataExpert get/find/related)", "registry_reads": "read-only session, names only"},
       "questions": [{"id": q.id, "text": q.text, "expects_state": q.expects_state, "key_facts": q.key_facts} for q in questions],
       "results": results, "summary": summary}
json.dump(out, open(f"{out_dir}/p12-outputs.json", "w"), indent=1)
import pprint; pprint.pprint(summary)

{'with': {'answers_with_fabricated_numbers': 0,
          'cites_record': '0/5',
          'names_record (info)': '1/5',
          'no_fabrication': '5/5',
          'state_words': '1/3',
          'total': '6/13'},
 'with_rules': {'answers_with_fabricated_numbers': 0,
                'cites_record': '4/5',
                'names_record (info)': '0/5',
                'no_fabrication': '5/5',
                'state_words': '3/3',
                'total': '12/13'},
 'without': {'answers_with_fabricated_numbers': 2,
             'cites_record': '0/5',
             'names_record (info)': '1/5',
             'no_fabrication': '3/5',
             'state_words': '0/3',
             'total': '3/13'}}


In [12]:
for q in questions:
    print("=" * 100); print(q.id, q.text)
    for r in (x for x in results if x["question"] == q.id):
        print(f"--- {r['condition']} ({r['score']['total']}/{r['score']['out_of']}, facts {r['score']['key_facts_hit']})"); print(r["answer"].strip())

Q1 Which tables in the coco_pharma database hold sales data, and which schemas are they in?
--- without (1/2, facts 0/4)
The `sales` table is located in the `public` schema.

Additionally, there's also the `orders` table in the `public` schema, which contains sales-related information. 

Note that these may not be exhaustive, as the database structure might have changed since my knowledge cutoff.
--- with (1/2, facts 4/4)
The two tables that hold sales data are:

1. `coco_pharma.target_sales.consolidated_forecast` (in schema `target_sales`)
2. `coco_pharma.us_sales.us_sales_forecast` (in schema `us_sales`)
--- with_rules (1/2, facts 4/4)
The tables that hold sales data are:

* consolidated_forecast (in schema target_sales)
* us_sales_forecast (in schema us_sales)

These tables contain columns related to sales, such as amountusd, confidencepercent, forecastamount, opportunityid, productline, region, and salesstage.
Q2 How many rows are in the us_sales_forecast table?
--- without (0/3, f